# 01 — Model A Standard CNN

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**  
**Owner:** Rajitha  
**Environment:** Google Colab + TensorFlow/Keras

## Goal
Build the standard Conv2D + MaxPooling baseline, train for at least 20 epochs, and export all required metrics.

In [ ]:
%pip install -q tensorflow-datasets scikit-learn

In [ ]:
import os, gc, time, json, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
print('TensorFlow:',tf.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
SEED=42
IMG_SIZE=64
BATCH_SIZE=64
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(SEED,IMG_SIZE,BATCH_SIZE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/EN3150_A03')
TFDS_ROOT=PROJECT_ROOT/'tfds_data'; ARTIFACT_ROOT=PROJECT_ROOT/'artifacts'; RESULT_ROOT=PROJECT_ROOT/'shared_results'; PLOT_ROOT=PROJECT_ROOT/'plots'
for p in [TFDS_ROOT,ARTIFACT_ROOT,RESULT_ROOT,PLOT_ROOT]: p.mkdir(parents=True,exist_ok=True)
print(PROJECT_ROOT)

In [ ]:
SPLITS=['train[:70%]','train[70%:85%]','train[85%:]']
(raw_train,raw_val,raw_test),ds_info=tfds.load('tf_flowers',split=SPLITS,as_supervised=True,with_info=True,data_dir=str(TFDS_ROOT),shuffle_files=False)
CLASS_NAMES=ds_info.features['label'].names
NUM_CLASSES=len(CLASS_NAMES)
def count_examples(ds): return int(tf.data.experimental.cardinality(ds).numpy())
print('Classes:',CLASS_NAMES)
print('Train:',count_examples(raw_train),'Val:',count_examples(raw_val),'Test:',count_examples(raw_test))

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
def preprocess(image,label):
    image=tf.image.resize(image,[IMG_SIZE,IMG_SIZE],antialias=True)
    return tf.cast(image,tf.float32),label
train_ds=(raw_train.shuffle(2048,seed=SEED,reshuffle_each_iteration=True).map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds=(raw_val.map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_ds=(raw_test.map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
chore: verify shared dataset setup for model A
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Model A architecture

In [ ]:
def build_model_a():
    inp=keras.Input((IMG_SIZE,IMG_SIZE,3),name='image'); x=layers.Rescaling(1/255.)(inp)
    x=layers.Conv2D(32,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.Conv2D(64,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.Conv2D(128,3,padding='same',activation='relu')(x); x=layers.MaxPooling2D()(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dense(64,activation='relu')(x); out=layers.Dense(NUM_CLASSES,name='logits')(x)
    return keras.Model(inp,out,name='Model_A_Standard_CNN')
model_a=build_model_a(); model_a.summary(); print('Parameters:',model_a.count_params())

In [ ]:
def parameter_table(model):
    rows=[]
    for l in model.layers:
        p=int(sum(np.prod(v.shape) for v in l.trainable_weights))
        if p: rows.append({'layer':l.name,'type':l.__class__.__name__,'output_shape':str(l.output.shape),'trainable_params':p})
    return pd.DataFrame(rows)
display(parameter_table(model_a))

In [ ]:
def estimate_macs_a(model):
    total=0
    for l in model.layers:
        if isinstance(l,layers.Conv2D):
            h,w,cout=map(int,l.output.shape[1:]); cin=int(l.input.shape[-1]); kh,kw=l.kernel_size; total+=h*w*cout*kh*kw*cin
        elif isinstance(l,layers.Dense): total+=int(l.input.shape[-1])*int(l.units)
    return int(total)
MODEL_A_MACS=estimate_macs_a(model_a); print('Approx MACs:',f'{MODEL_A_MACS:,}')

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
feat: implement standard CNN model A architecture
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Train 20 epochs with resume protection

In [ ]:
LOSS_FN=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
class PersistentEpochTimer(keras.callbacks.Callback):
    def __init__(self,csv_path): super().__init__(); self.csv_path=Path(csv_path); self.csv_path.parent.mkdir(parents=True,exist_ok=True)
    def on_epoch_begin(self,epoch,logs=None): self.start=time.perf_counter()
    def on_epoch_end(self,epoch,logs=None):
        row=pd.DataFrame([{'epoch':int(epoch),'seconds':float(time.perf_counter()-self.start)}])
        row.to_csv(self.csv_path,mode='a',header=not self.csv_path.exists(),index=False)
def read_log(run_name):
    p=ARTIFACT_ROOT/run_name/'training_log.csv'
    if not p.exists(): return pd.DataFrame()
    d=pd.read_csv(p)
    return d.drop_duplicates(subset=['epoch'],keep='last').sort_values('epoch') if 'epoch' in d else d
def average_epoch_time(run_name):
    p=ARTIFACT_ROOT/run_name/'epoch_times.csv'
    if not p.exists(): return np.nan
    d=pd.read_csv(p).drop_duplicates(subset=['epoch'],keep='last')
    return float(d.seconds.mean()) if len(d) else np.nan
def fit_resumable(model,run_name,optimizer,epochs):
    rd=ARTIFACT_ROOT/run_name; rd.mkdir(parents=True,exist_ok=True)
    final=rd/'final.keras'; best=rd/'best.keras'; backup=rd/'backup'; log=rd/'training_log.csv'; timing=rd/'epoch_times.csv'
    if final.exists(): print('Completed run found:',run_name); return keras.models.load_model(final)
    model.compile(optimizer=optimizer,loss=LOSS_FN,metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    callbacks=[keras.callbacks.BackupAndRestore(backup_dir=str(backup),save_freq='epoch',delete_checkpoint=False),keras.callbacks.ModelCheckpoint(str(best),monitor='val_accuracy',mode='max',save_best_only=True,verbose=1),keras.callbacks.CSVLogger(str(log),append=True),PersistentEpochTimer(timing)]
    model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=1)
    model.save(final); return model
def reset_run(run_name):
    import shutil
    p=ARTIFACT_ROOT/run_name
    if p.exists(): shutil.rmtree(p)
def plot_history(run_name,prefix):
    d=read_log(run_name)
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.loss,label='Train'); plt.plot(d.epoch+1,d.val_loss,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(prefix+' Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_loss.png'),dpi=180); plt.show()
    plt.figure(figsize=(8,5)); plt.plot(d.epoch+1,d.accuracy,label='Train'); plt.plot(d.epoch+1,d.val_accuracy,label='Validation'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(prefix+' Accuracy'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/(prefix.lower().replace(' ','_')+'_accuracy.png'),dpi=180); plt.show()

In [ ]:
model_a=fit_resumable(build_model_a(),'model_a_adam',keras.optimizers.Adam(learning_rate=1e-3),20)
plot_history('model_a_adam','Model A')

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
exp: train model A for 20 epochs and add learning curves
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

## Test evaluation

In [ ]:
def get_true_labels(ds): return np.concatenate([y.numpy() for _,y in ds])
Y_TEST=get_true_labels(test_ds)
def model_file_size_mb(path): return Path(path).stat().st_size/(1024**2)
def benchmark_inference_ms_per_image(model,ds,max_batches=10):
    batches=[]; n=0
    for i,(x,_) in enumerate(ds):
        if i>=max_batches: break
        batches.append(x); n+=int(x.shape[0])
    if not batches: return np.nan
    _=model(batches[0],training=False); t=time.perf_counter()
    for x in batches: _=model(x,training=False)
    return (time.perf_counter()-t)*1000/n
def evaluate_model(model,name,save_name):
    pred=np.argmax(model.predict(test_ds,verbose=0),axis=1)
    out={'accuracy':float(accuracy_score(Y_TEST,pred)),'precision_macro':float(precision_score(Y_TEST,pred,average='macro',zero_division=0)),'recall_macro':float(recall_score(Y_TEST,pred,average='macro',zero_division=0))}
    print(out); print(classification_report(Y_TEST,pred,target_names=CLASS_NAMES,digits=4,zero_division=0))
    disp=ConfusionMatrixDisplay(confusion_matrix(Y_TEST,pred),display_labels=CLASS_NAMES); disp.plot(xticks_rotation=45); plt.title(name+' — Confusion Matrix'); plt.tight_layout(); plt.savefig(PLOT_ROOT/save_name,dpi=180); plt.show(); return out

In [ ]:
best=keras.models.load_model(ARTIFACT_ROOT/'model_a_adam'/'best.keras'); m=evaluate_model(best,'Model A','model_a_confusion_matrix.png'); log=read_log('model_a_adam')
res={'model':'Model A — Standard CNN','owner':'Rajitha','optimizer':'Adam (lr=0.001)','parameters':int(best.count_params()),'trainable_parameters':int(sum(np.prod(v.shape) for v in best.trainable_weights)),'model_size_mb':float(model_file_size_mb(ARTIFACT_ROOT/'model_a_adam'/'final.keras')),'estimated_fp32_weight_kb':float(best.count_params()*4/1024),'best_val_accuracy':float(log.val_accuracy.max()),'accuracy':m['accuracy'],'precision_macro':m['precision_macro'],'recall_macro':m['recall_macro'],'avg_epoch_time_s':float(average_epoch_time('model_a_adam')),'inference_ms_per_image':float(benchmark_inference_ms_per_image(best,test_ds)),'approx_macs':int(MODEL_A_MACS)}
json.dump(res,open(RESULT_ROOT/'model_a.json','w'),indent=2); print(json.dumps(res,indent=2))

## Interpretation
Write a short paragraph about overfitting/underfitting from the curves, test accuracy, main class confusions, parameter/model size, epoch time, and why this is the baseline for Model B.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
analysis: add model A test metrics confusion matrix and result export
```
File:
```text
notebooks/01_model_a_standard.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.